# AFSP regenerated under the corrected centroid (val)

---
## 1 — Host, working tree, disk

In [11]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

name, memory.total [MiB], memory.used [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 3 MiB, 590.48.01


In [12]:
from pathlib import Path

if not Path("manage.py").exists():
    if not Path("Style-Aware-MT/manage.py").exists():
        !git clone -b fix/marker-casefold https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT

!git fetch origin
!git checkout fix/marker-casefold
!git pull --ff-only origin fix/marker-casefold
!git rev-parse --short HEAD

Already on 'fix/marker-casefold'
Your branch is up to date with 'origin/fix/marker-casefold'.
From https://github.com/prnamhr/Style-Aware-MT
 * branch            fix/marker-casefold -> FETCH_HEAD
Already up to date.
dbf20f0


In [3]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 56.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 235.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 185.5 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 184.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 71.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 111.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 187.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 139.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 134.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 230.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 212.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 98.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1

In [4]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

NVIDIA GeForce RTX 4090  sm_89 torch 2.12.0+cu130 / cuda 13.0


---
## 2 — Run parameters

Both conditions share one `retrieval:`/`afsp:`/`prompt:` block; the configs differ only in
`generator.adapter_path`. Exemplar selection is therefore identical between them, which is
why section 4 probes it once.

In [13]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone

import yaml

SPLIT = 'val'
CASEFIX_COMMIT = '141c532'          # where the marker regex became case-folding
CENTROID = Path('results/stylometrics_centroid.json')
CORRECTED_FINGERPRINT = 'fd5aec8d69454b02'

# (condition to run, config, output stem). --condition stays a known rung; --out-name
# is what keeps the corrected pass beside the old one instead of overwriting it.
RUNS = [
    ('afsp_full', Path('configs/base_qwen.yaml'), 'afsp_full_casefix'),
    ('peft_afsp', Path('configs/peft_afsp.yaml'), 'peft_afsp_casefix'),
]

CFGS = {c: yaml.safe_load(p.read_text(encoding='utf-8')) for c, p, _ in RUNS}
CFG = CFGS['afsp_full']
RETR, AFSP, PROMPT = CFG['retrieval'], CFG['afsp'], CFG['prompt']

for blk in ('retrieval', 'afsp', 'prompt', 'data'):
    assert CFGS['afsp_full'][blk] == CFGS['peft_afsp'][blk], f'{blk} differs across the two configs'

for cond, path, _ in RUNS:
    gen = CFGS[cond]['generator']
    assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
    assert (gen['temperature'], gen['top_p']) == (0.0, 1.0), 'not the locked greedy decoding'
    assert (gen['max_tokens'], gen['seed']) == (1024, 42), gen
    assert gen['dtype'] == 'bfloat16' and gen['load_in_4bit'] is False, 'quantizing redefines the base'
    assert CFGS[cond]['data']['eval_file'] == f'data/splits/{SPLIT}.jsonl', CFGS[cond]['data']

# The frozen operating point (DEVLOG 2026-07-23), confirmed unmoved by afsp_sweep --score-only.
assert RETR['k'] == 8, RETR
assert (AFSP['beta'], AFSP['lambda_style']) == (0.3, 0.75), AFSP
assert (AFSP['style_objective'], AFSP['style_target_sigma']) == ('bandpass', 1.0), AFSP
assert AFSP['centroid_file'] == str(CENTROID), AFSP['centroid_file']

ADAPTER = Path(CFGS['peft_afsp']['generator']['adapter_path'])
assert str(ADAPTER) == 'models/peft_lora_r32_lr2e-4/checkpoint-1358', ADAPTER

# Wall-clock end of the booking; a duration from this cell would ignore setup already spent.
BOOKING_END = '2026-08-21T21:00:00+03:00'  # e.g. '2026-08-21T21:00+00:00'
assert BOOKING_END, 'set BOOKING_END to the wall-clock end of the GPU booking'
DEADLINE = datetime.fromisoformat(BOOKING_END)
assert DEADLINE.tzinfo is not None, 'BOOKING_END needs an explicit UTC offset'
assert DEADLINE > datetime.now(timezone.utc), f'the booking ended at {BOOKING_END}'

print(f'k={RETR["k"]} lambda={AFSP["lambda_style"]} beta={AFSP["beta"]} '
      f'sigma={AFSP["style_target_sigma"]}  ->  {[n for _, _, n in RUNS]}')
print(f'booking ends {DEADLINE:%Y-%m-%d %H:%M %Z}, '
      f'{(DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600:.1f} h from now')

k=8 lambda=0.75 beta=0.3 sigma=1.0  ->  ['afsp_full_casefix', 'peft_afsp_casefix']
booking ends 2026-08-21 21:00 UTC+03:00, 11.9 h from now


In [14]:
from src.eval.stylometrics import fingerprint

CENTROID_NEW = json.loads(CENTROID.read_text(encoding='utf-8'))
FP = fingerprint(CENTROID_NEW)
assert FP == CORRECTED_FINGERPRINT, (
    f'{CENTROID} fingerprints to {FP}, not the corrected {CORRECTED_FINGERPRINT}. '
    f'This run would bake a third centroid into the outputs.')

mr = CENTROID_NEW['features'].index('marker_rate')
print(f'centroid {FP}  marker_rate mean={CENTROID_NEW["mean"][mr]:.6f} '
      f'std={CENTROID_NEW["std"][mr]:.6f}')

centroid fd5aec8d69454b02  marker_rate mean=0.057348 std=0.078181


In [15]:
import getpass
import logging

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

HF_TOKEN set, no rater keys present


---
## 3 — Weights and the index, before the GPU is touched

Network first. Every download that happens after the base is resident is a GPU-hour spent
waiting on bandwidth.

In [16]:
import hashlib

from huggingface_hub import snapshot_download

HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')

if not all((ADAPTER / f).exists() for f in ADAPTER_FILES):
    snapshot_download(HF_REPO, local_dir='.', token=os.environ['HF_TOKEN'],
                      allow_patterns=[f'{ADAPTER}/{f}' for f in ADAPTER_FILES])

conf = json.loads((ADAPTER / 'adapter_config.json').read_text(encoding='utf-8'))
assert conf['r'] == 32 and conf['lora_alpha'] == 64, conf
assert conf['base_model_name_or_path'].endswith(CFG['generator']['model'].split('/')[-1]), conf
ADAPTER_SHA = hashlib.sha256((ADAPTER / 'adapter_model.safetensors').read_bytes()).hexdigest()
print(f'{ADAPTER}  r={conf["r"]} alpha={conf["lora_alpha"]}  {ADAPTER_SHA[:12]}')

models/peft_lora_r32_lr2e-4/checkpoint-1358  r=32 alpha=64  ad97c46af852


In [17]:
t0 = time.perf_counter()
snapshot_download(CFG['generator']['model'],
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'],
                  token=os.environ['HF_TOKEN'], max_workers=8)
print(f"base cached in {(time.perf_counter() - t0) / 60:.1f} min")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

base cached in 0.8 min


In [21]:
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
INDEX_REBUILT = not all((INDEX / f).exists() for f in INDEX_FILES)

if INDEX_REBUILT:
    print('index missing -- rebuilding; exemplar selection may differ from the reported AFSP run')
    !python3 manage.py build_index --config configs/base_qwen.yaml

INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest() for f in INDEX_FILES}
meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'] and meta['indexed_side'] == 'source', meta
assert meta['n_passages'] == 10860, meta
print(f'index {"rebuilt" if INDEX_REBUILT else "transferred"}, {meta["n_passages"]} passages')
for f, digest in INDEX_SHA.items():
    print(f'  {f:16s} {digest[:12]}')

index transferred, 10860 passages
  embeddings.npy   9c282c8042ab
  pairs.jsonl      c48f42980943
  meta.json        b028a2a81f9f


---
## 4 — The gate: does the corrected centroid move exemplar selection?


In [22]:
from src.retrieval.afsp import AFSPRetriever
from src.retrieval.retrieve import RetrievalIndex

ROWS = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
VAL_SRC = [r['input'] for r in ROWS]
print(f'{len(ROWS)} {SPLIT} segments')

CENTROID_OLD = json.loads(subprocess.run(
    ['git', 'show', f'{CASEFIX_COMMIT}^:results/stylometrics_centroid.json'],
    capture_output=True, text=True, check=True).stdout)
FP_OLD = fingerprint(CENTROID_OLD)
assert FP_OLD != FP, 'the two centroids are the same file; there is nothing to regenerate'
print(f'old {FP_OLD}  marker_rate mean={CENTROID_OLD["mean"][mr]:.6f} '
      f'std={CENTROID_OLD["std"][mr]:.6f}')

1323 val segments
old 5a3e703bfd82da17  marker_rate mean=0.032684 std=0.056741


In [23]:
index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])

def reranker(centroid):
    """Exactly how src.infer.run._select_afsp builds it for a rerank condition."""
    return AFSPRetriever(
        index, centroid, index_dir=RETR['index_dir'],
        beta=AFSP['beta'], knn_hubness=AFSP['knn_hubness'], pool_mult=AFSP['pool_mult'],
        lambda_style=AFSP['lambda_style'], style_objective=AFSP['style_objective'],
        style_target_sigma=AFSP['style_target_sigma'],
        style_register_direction=AFSP['style_register_direction'])

t0 = time.perf_counter()
SEL_OLD = reranker(CENTROID_OLD).select(VAL_SRC, k=RETR['k'])
SEL_NEW = reranker(CENTROID_NEW).select(VAL_SRC, k=RETR['k'])
print(f'both selections in {time.perf_counter() - t0:.0f}s')

Loading intfloat/multilingual-e5-large-instruct on device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

both selections in 18s


In [24]:
old_sets = [tuple(e['input'] for e in row) for row in SEL_OLD]
new_sets = [tuple(e['input'] for e in row) for row in SEL_NEW]

changed_set = [i for i, (a, b) in enumerate(zip(old_sets, new_sets)) if set(a) != set(b)]
changed_order = [i for i, (a, b) in enumerate(zip(old_sets, new_sets))
                 if set(a) == set(b) and a != b]
overlap = [len(set(a) & set(b)) for a, b in zip(old_sets, new_sets)]

SELECTION_DIVERGENCE = {
    'n_segments': len(ROWS),
    'k': RETR['k'],
    'changed_set': len(changed_set),
    'changed_order_only': len(changed_order),
    'identical': len(ROWS) - len(changed_set) - len(changed_order),
    'mean_exemplars_shared': round(sum(overlap) / len(overlap), 3),
    'centroid_old': FP_OLD,
    'centroid_new': FP,
}
for key, val in SELECTION_DIVERGENCE.items():
    print(f'  {key:24s} {val}')

  n_segments               1323
  k                        8
  changed_set              967
  changed_order_only       311
  identical                45
  mean_exemplars_shared    6.921
  centroid_old             5a3e703bfd82da17
  centroid_new             fd5aec8d69454b02


In [25]:
share = SELECTION_DIVERGENCE['changed_set'] / len(ROWS)
assert SELECTION_DIVERGENCE['changed_set'] or SELECTION_DIVERGENCE['changed_order_only'], (
    'Selection is unchanged under the corrected centroid. Regeneration buys nothing -- '
    'stop here and keep the rescored artifacts.')

print(f'{share:.1%} of prompts get a different exemplar set under the corrected '
      f'centroid, {SELECTION_DIVERGENCE["changed_order_only"]} more get the same set in a '
      f'different order. Rescoring cannot reach this. Section 6 may run.')

73.1% of prompts get a different exemplar set under the corrected centroid, 311 more get the same set in a different order. Rescoring cannot reach this. Section 6 may run.


---
## 5 — Throughput and fit

In [26]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, make_client, order_exemplars

STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)

PROBE_N = 8
probe = [build_fewshot_user(s, order_exemplars(ex, PROMPT['ordering']), GLOSSARY)
         for s, ex in zip(VAL_SRC[:PROBE_N], SEL_NEW[:PROBE_N])]

# The adapter pass is the slower of the two, so it bounds both; probing it here also
# forces PeftModel.from_pretrained before section 6 rather than 25 minutes into it.
PROBE_GEN = CFGS['peft_afsp']['generator']
t0 = time.perf_counter()
client = make_client(PROBE_GEN)
load_s = time.perf_counter() - t0

t0 = time.perf_counter()
for user in probe:
    client.complete(STYLE, user)
seg_s = (time.perf_counter() - t0) / PROBE_N

print(f'{load_s:.0f}s load with {Path(PROBE_GEN["adapter_path"]).name}, '
      f'{seg_s:.2f}s per segment at k={RETR["k"]}')
print(f'{torch.cuda.max_memory_allocated() / 2**30:.1f} GiB peak')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


7s load with checkpoint-1358, 1.91s per segment at k=8
16.1 GiB peak


In [27]:
# seg_s came off the adapter, so this is an upper bound on the base pass too.
pass_h = len(ROWS) * seg_s / 3600
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{pass_h:.2f} h per pass, {pass_h * len(RUNS):.1f} h for {len(RUNS)}, '
      f'{left_h:.1f} h left of the booking')

assert pass_h * len(RUNS) <= 0.9 * left_h, (
    f'{pass_h * len(RUNS):.1f} h of generation does not fit the {left_h:.1f} h left of the '
    f'booking at 90% occupancy. Shorten the run or rebook.')

print('\nFits. Section 6 may start.')

0.70 h per pass, 1.4 h for 2, 11.8 h left of the booking

Fits. Section 6 may start.


In [30]:
torch.cuda.empty_cache()

---
## 6 — Generation

In [31]:
TIMING = {}
for cond, config, name in RUNS:
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', str(config), '--out-name', name], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

afsp_full: selecting k=8 for 1323 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 190.12it/s]


Output name overridden: condition 'afsp_full' -> outputs/afsp_full_casefix_val.jsonl
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (afsp_full) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  365/1323
  370/1323
  375/1323
  380/1323
  385/1323
  390/1323
  

Loading weights: 100%|██████████| 339/339 [00:01<00:00, 188.98it/s]
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


Output name overridden: condition 'peft_afsp' -> outputs/peft_afsp_casefix_val.jsonl
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (peft_afsp) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  365/1323
  370/1323
  375/1323
  380/1323
  385/1323
  390/1323
  

---
## 7 — The outputs

`--out-name` changes the filename, not the `condition` field written into each row: the
rows still say `afsp_full`/`peft_afsp`, which is correct -- the method is the same, only
the centroid it read has been fixed.

In [32]:
for cond, _, name in RUNS:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(ROWS), f'{name}: {len(rows)} rows, expected {len(ROWS)}'
    assert [r['input'] for r in rows] == VAL_SRC, f'{name}: source order differs from {SPLIT}.jsonl'
    assert all(r['condition'] == cond for r in rows), f'{name}: mislabelled rows'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    old = Path(f'outputs/{cond}_{SPLIT}.jsonl')
    moved = sum(1 for a, b in zip(rows, [json.loads(x) for x in old.open(encoding='utf-8') if x.strip()])
                if a['prediction'] != b['prediction']) if old.exists() else None
    caveat = ' (index rebuilt here -- not the index that run read)' if INDEX_REBUILT else ''
    print(f'{name}: {len(rows)} rows, {len(blank)} blank {blank[:5]}, '
          f'{moved} translations differ from {cond}{caveat}')

afsp_full_casefix: 1323 rows, 0 blank [], 1132 translations differ from afsp_full
peft_afsp_casefix: 1323 rows, 0 blank [], 953 translations differ from peft_afsp


In [33]:
# The corrected pass must record the corrected centroid, not inherit the old provenance.
for _, _, name in RUNS:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    prov = usage['provenance']
    assert prov['centroid']['fingerprint'] == CORRECTED_FINGERPRINT, prov
    assert prov['lambda_style'] == AFSP['lambda_style'] and prov['k'] == RETR['k'], prov
    print(f'{name}: {json.dumps(prov)}')

afsp_full_casefix: {"k": 8, "ordering": "most_similar_last", "index_dir": "data/knn_index", "beta": 0.3, "lambda_style": 0.75, "style_objective": "bandpass", "style_target_sigma": 1.0, "centroid": {"path": "results/stylometrics_centroid.json", "fingerprint": "fd5aec8d69454b02"}}
peft_afsp_casefix: {"adapter_path": "models/peft_lora_r32_lr2e-4/checkpoint-1358", "k": 8, "ordering": "most_similar_last", "index_dir": "data/knn_index", "beta": 0.3, "lambda_style": 0.75, "style_objective": "bandpass", "style_target_sigma": 1.0, "centroid": {"path": "results/stylometrics_centroid.json", "fingerprint": "fd5aec8d69454b02"}}


---
## 8 — Scoring on the host


In [42]:
NEW_CONDS = [n for _, _, n in RUNS]
# Each corrected stem beside the run it supersedes, plus the plain-retrieval control.
EVAL_CONDS = ' '.join([c for cond, _, name in RUNS for c in (cond, name)] + ['peft_knn'])
!python3 manage.py eval --split {SPLIT} --conditions {EVAL_CONDS}

condition          n     BLEU   chrF   marker_rate  ref_marker_rate
-------------------------------------------------------------------
afsp_full          1323  14.52  39.99  1.2          0.93           
afsp_full_casefix  1323  14.17  39.88  1.26         0.93           
peft_afsp          1323  17.77  42.12  0.95         0.93           
peft_afsp_casefix  1323  17.85  42.17  0.99         0.93           
peft_knn           1323  17.98  42.4   1.01         0.93           


In [43]:
CI_CONDS = ' '.join(['knn_fewshot', 'peft', 'peft_knn']
                    + [c for cond, _, name in RUNS for c in (cond, name)])
CI_PATH = f'results/stylometrics_ci_casefix_{SPLIT}.json'
!python3 manage.py stylometrics_ci --split {SPLIT} --conditions {CI_CONDS} --results_path {CI_PATH}


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition          stylo_dist  ci95              P(this rank)  modal rank  mean rank
------------------------------------------------------------------------------------------
1     peft               0.2890      [0.2472, 0.3420]  0.545         1 (0.545)   1.74     
2     afsp_full_casefix  0.2961      [0.2555, 0.3462]  0.350         2 (0.350)   2.18     
3     afsp_full          0.3032      [0.2588, 0.3542]  0.353         3 (0.353)   2.69     
4     peft_afsp          0.3244      [0.2817, 0.3739]  0.474         4 (0.474)   4.07     
5     peft_afsp_casefix  0.3290      [0.2851, 0.3803]  0.501         5 (0.501)   4.46     
6     peft_knn           0.3589      [0.3151, 0.4092]  0.575         6 (0.575)   6.33     
7     knn_fewshot        0.3659      [0.3214, 0.4177]  0.616         7 (0.616)   6.52     

Si

In [44]:
# What the regeneration actually bought, on the register axis.
ci = json.loads(Path(CI_PATH).read_text(encoding='utf-8'))
assert ci['centroid']['fingerprint'] == CORRECTED_FINGERPRINT, ci['centroid']
# stylometrics_ci skips a condition whose file is missing rather than failing.
missing = [n for n in NEW_CONDS if n not in ci['conditions']]
assert not missing, f'{missing} were skipped; the report does not contain the new run'
print(f"{'condition':22s} {'stylo_dist':>10s}  rank")
for cond in ci['ranking']:
    print(f'{cond:22s} {ci["cells"][cond]["stylo_dist"]:>10.4f}  {ci["cells"][cond]["rank"]}')

print()
for pair in (('afsp_full', 'afsp_full_casefix'), ('peft_afsp', 'peft_afsp_casefix')):
    hit = [p for p in ci['paired_all'] if {p['a'], p['b']} == set(pair)]
    for p in hit:
        print(f'{p["a"]} - {p["b"]}: {p["diff"]:+.4f} [{p["ci_low"]:+.4f}, {p["ci_high"]:+.4f}] '
              f'p={p["p_value"]:.4f}')

condition              stylo_dist  rank
peft                       0.2890  1
afsp_full_casefix          0.2961  2
afsp_full                  0.3032  3
peft_afsp                  0.3244  4
peft_afsp_casefix          0.3290  5
peft_knn                   0.3589  6
knn_fewshot                0.3659  7

afsp_full_casefix - afsp_full: -0.0064 [-0.0374, +0.0242] p=0.6940
peft_afsp - peft_afsp_casefix: -0.0042 [-0.0273, +0.0196] p=0.7300


---
## 9 — Manifest and bundle

In [45]:
import platform

import peft as peft_lib
import transformers

MANIFEST = {
    'purpose': 'AFSP regenerated with exemplars reranked against the corrected centroid',
    'runs': [{'condition': c, 'config': str(p), 'output_name': n} for c, p, n in RUNS],
    'split': SPLIT,
    'supersedes': ['afsp_full', 'peft_afsp'],
    'centroid': {'path': str(CENTROID), 'fingerprint': FP, 'superseded_fingerprint': FP_OLD,
                 'casefix_commit': CASEFIX_COMMIT},
    'selection_divergence': SELECTION_DIVERGENCE,
    'selection_divergence_basis': (
        'Both selections are reconstructions run in this session over the index digested '
        'under `index` below. The exemplars the committed afsp_full/peft_afsp runs actually '
        'received are stored neither in their output rows nor in their usage sidecars, which '
        'predate the provenance block. The divergence therefore compares two centroids over '
        "today's index, and carries over to the committed runs only if that index is "
        'byte-identical to the one they read -- which index.rebuilt_here must be false for. '
        'The same condition governs the per-condition counts of moved translations.'),
    'generator': {c: CFGS[c]['generator'] for c, _, _ in RUNS},
    'retrieval': {'k': RETR['k'], 'embed_model': RETR['embed_model'], 'index_dir': str(INDEX)},
    'afsp': {k: AFSP[k] for k in ('beta', 'lambda_style', 'style_objective',
                                  'style_target_sigma', 'pool_mult', 'knn_hubness')},
    'adapter': {'path': str(ADAPTER), 'sha256': ADAPTER_SHA,
                'r': conf['r'], 'lora_alpha': conf['lora_alpha']},
    'index': {'rebuilt_here': INDEX_REBUILT, 'sha256': INDEX_SHA, 'meta': meta},
    'timing': TIMING,
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip(),
    'versions': {
        'device': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'transformers': transformers.__version__,
        'peft': peft_lib.__version__,
        'python': platform.python_version(),
    },
}
out = Path('outputs/afsp_casefix_manifest.json')
out.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(out, out.stat().st_size, 'bytes')

outputs/afsp_casefix_manifest.json 3637 bytes


In [46]:
import zipfile

# Everything the local pass needs; the host keeps nothing.
BUNDLE = Path(f'afsp_casefix_{SPLIT}.zip')
FILES = ([Path(f'outputs/{n}_{SPLIT}.jsonl') for n in NEW_CONDS]
         + [Path(f'outputs/{n}_{SPLIT}_usage.json') for n in NEW_CONDS]
         + [out, Path(CI_PATH)])
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        assert f.exists(), f
        z.write(f, str(f))
print(f'{BUNDLE}  {BUNDLE.stat().st_size / 1e6:.1f} MB  ({len(FILES)} files)')

afsp_casefix_val.zip  0.4 MB  (6 files)


---
## 10 — Seal

In [47]:
# 1. Nothing was spent.
for _, _, name in RUNS:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, usage
    print(f'{name}: {usage["calls"]} calls, ${usage.get("cost_usd", 0.0):.2f}')

# 2. The test split was not touched.
assert not list(Path('outputs').glob('*_test.jsonl')), 'a test-split output exists'
assert not list(Path('results').glob('*_test.json')), 'a test-split result exists'

# 3. No rater key was ever present in this session.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var

# 4. The superseded outputs are untouched -- this session adds, it does not overwrite.
!git status --short outputs/afsp_full_{SPLIT}.jsonl outputs/peft_afsp_{SPLIT}.jsonl

print('\nsealed: 0 paid calls, test split untouched, no rater key present')

afsp_full_casefix: 1323 calls, $0.00
peft_afsp_casefix: 1323 calls, $0.00

sealed: 0 paid calls, test split untouched, no rater key present


In [ ]:
!git status --short outputs results